In [1]:
import pandas as pd
import sqlite3
import os
import matplotlib.pyplot as plt

os.makedirs("../database", exist_ok=True)

In [2]:
df = pd.read_csv("../data/SampleSuperstore_Cleaned.csv")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
conn = sqlite3.connect("../database/superstore.db")

In [4]:
df.to_sql(
    "Orders",
    conn,
    if_exists="replace",
    index=False
)

9994

In [5]:
pd.read_sql("""
SELECT *
FROM Orders
LIMIT 5;
""", conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [6]:
def save_query_result(query, filename):
    # Run SQL query
    result = pd.read_sql(query, conn)

    # Display result
    display(result)

    # Save CSV
    result.to_csv(f"../results/{filename}.csv", index=False)

    # Save as image
    fig, ax = plt.subplots(figsize=(10, max(1, len(result)*0.35)))

    ax.axis('off')

    table = ax.table(
        cellText=result.values,
        colLabels=result.columns,
        loc='center'
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.3)

    plt.savefig(
        f"../images/{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"✅ Saved {filename}.csv")
    print(f"✅ Saved {filename}.png")

In [7]:
query = """
SELECT
ROUND(SUM(Sales),2) AS TotalSales
FROM Orders;
"""

save_query_result(query,"total_sales")
pd.read_sql(query, conn)

,TotalSales
0,2297200.86


✅ Saved total_sales.csv
✅ Saved total_sales.png


,TotalSales
0,2297200.86


In [8]:
query = """
SELECT
ROUND(SUM(Profit),2) AS TotalProfit
FROM Orders;
"""
save_query_result(query,"total_profit")
pd.read_sql(query, conn)

,TotalProfit
0,286397.02


✅ Saved total_profit.csv
✅ Saved total_profit.png


,TotalProfit
0,286397.02


In [9]:
query = """
SELECT
COUNT(*) AS TotalOrders
FROM Orders;
"""

save_query_result(query,"total_orders")
pd.read_sql(query, conn)

,TotalOrders
0,9994


✅ Saved total_orders.csv
✅ Saved total_orders.png


,TotalOrders
0,9994


In [10]:
query = """
SELECT
Category,
ROUND(SUM(Sales),2) AS Sales
FROM Orders
GROUP BY Category
ORDER BY Sales DESC;
"""

save_query_result(query,"sales_by_category")
pd.read_sql(query, conn)

,Category,Sales
0,Technology,836154.03
1,Furniture,741999.80
2,Office Supplies,719047.03


✅ Saved sales_by_category.csv
✅ Saved sales_by_category.png


,Category,Sales
0,Technology,836154.03
1,Furniture,741999.80
2,Office Supplies,719047.03


In [11]:
query = """
SELECT
Region,
ROUND(SUM(Profit),2) AS Profit
FROM Orders
GROUP BY Region
ORDER BY Profit DESC;
"""

save_query_result(query,"profit_by_region")
pd.read_sql(query, conn)

,Region,Profit
0,West,108418.45
1,East,91522.78
2,South,46749.43
3,Central,39706.36


✅ Saved profit_by_region.csv
✅ Saved profit_by_region.png


,Region,Profit
0,West,108418.45
1,East,91522.78
2,South,46749.43
3,Central,39706.36


In [12]:
query = """
SELECT
"Customer Name" AS CustomerName,
ROUND(SUM(Sales),2) AS Revenue
FROM Orders
GROUP BY "Customer Name"
ORDER BY Revenue DESC
LIMIT 10;
"""

save_query_result(query,"top_customers")
pd.read_sql(query, conn)

,CustomerName,Revenue
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
5,Ken Lonsdale,14175.23
6,Sanjit Chand,14142.33
7,Hunter Lopez,12873.30
8,Sanjit Engle,12209.44
9,Christopher Conant,12129.07


✅ Saved top_customers.csv
✅ Saved top_customers.png


,CustomerName,Revenue
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
5,Ken Lonsdale,14175.23
6,Sanjit Chand,14142.33
7,Hunter Lopez,12873.30
8,Sanjit Engle,12209.44
9,Christopher Conant,12129.07


In [13]:
import pandas as pd

query = """
SELECT
"Product Name" AS ProductName,
ROUND(SUM(Sales),2) AS Sales
FROM Orders
GROUP BY "Product Name"
ORDER BY Sales DESC
LIMIT 10;
"""
save_query_result(query, "top_products")

,ProductName,Sales
0,Canon imageCLASS 2200 Advanced Copier,61599.82
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48
3,HON 5400 Series Task Chairs for Big and Tall,21870.58
4,GBC DocuBind TL300 Electric Binding System,19823.48
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50
6,Hewlett Packard LaserJet 3310 Copier,18839.69
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90
8,GBC DocuBind P400 Electric Binding System,17965.07
9,High Speed Automatic Electric Letter Opener,17030.31


✅ Saved top_products.csv
✅ Saved top_products.png


In [ ]:
query = """
SELECT *
FROM Orders
WHERE Profit < 0;
"""

save_query_result(query, "negative_profit_orders")

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
1,15,US-2015-118983,2015-11-22,2015-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,76106,Central,OFF-AP-10002311,Office Supplies,Appliances,Holmes Replacement Filter for HEPA Air Cleaner...,68.8100,5,0.80,-123.8580
2,16,US-2015-118983,2015-11-22,2015-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,76106,Central,OFF-BI-10000756,Office Supplies,Binders,Storex DuraTech Recycled Plastic Frosted Binders,2.5440,3,0.80,-3.8160
3,24,US-2017-156909,2017-07-16,2017-07-18,Second Class,SF-20065,Sandra Flanagan,Consumer,United States,Philadelphia,...,19140,East,FUR-CH-10002774,Furniture,Chairs,"Global Deluxe Stacking Chair, Gray",71.3720,2,0.30,-1.0196
4,28,US-2015-150630,2015-09-17,2015-09-21,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,...,19140,East,FUR-BO-10004834,Furniture,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.4300,7,0.50,-1665.0522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1866,9921,CA-2016-149272,2016-03-15,2016-03-19,Standard Class,MY-18295,Muhammed Yedwab,Corporate,United States,Bryan,...,77803,Central,OFF-BI-10004233,Office Supplies,Binders,"GBC Pre-Punched Binding Paper, Plastic, White,...",22.3860,7,0.80,-35.8176
1867,9922,CA-2014-111360,2014-11-24,2014-11-30,Standard Class,AT-10435,Alyssa Tate,Home Office,United States,Akron,...,44312,East,OFF-BI-10003350,Office Supplies,Binders,Acco Expandable Hanging Binders,5.7420,3,0.70,-4.5936
1868,9932,CA-2015-104948,2015-11-13,2015-11-17,Standard Class,KH-16510,Keith Herrera,Consumer,United States,San Bernardino,...,92404,West,FUR-BO-10004357,Furniture,Bookcases,O'Sullivan Living Dimensions 3-Shelf Bookcases,683.3320,4,0.15,-40.1960
1869,9938,CA-2016-164889,2016-06-03,2016-06-06,Second Class,CP-12340,Christine Phan,Corporate,United States,Los Angeles,...,90049,West,FUR-TA-10001676,Furniture,Tables,Hon 61000 Series Interactive Training Tables,71.0880,2,0.20,-1.7772


C:\Users\user\AppData\Local\Temp\ipykernel_27108\2373868330.py:26: UserWarning: Glyph 147 (\x93) missing from font(s) DejaVu Sans.
  plt.savefig(
C:\Users\user\AppData\Local\Temp\ipykernel_27108\2373868330.py:26: UserWarning: Glyph 148 (\x94) missing from font(s) DejaVu Sans.
  plt.savefig(


In [14]:
query = """
SELECT
"Product Name",
Sales
FROM Orders
WHERE Sales >
(
SELECT AVG(Sales)
FROM Orders
);
"""
save_query_result(query,"above_average_sales")
pd.read_sql(query, conn)

,Product Name,Sales
0,Bush Somerset Collection Bookcase,261.9600
1,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,Bretford CR4500 Series Slim Rectangular Table,957.5775
3,Mitel 5320 IP Phone VoIP phone,907.1520
4,Chromcraft Rectangular Conference Tables,1706.1840
...,...,...
2355,Avaya 5410 Digital phone,271.9600
2356,Cisco SPA301,249.5840
2357,Ibico Recycled Linen-Style Covers,437.4720
2358,Aastra 57i VoIP phone,258.5760


C:\Users\user\AppData\Local\Temp\ipykernel_8624\2373868330.py:26: UserWarning: Glyph 147 (\x93) missing from font(s) DejaVu Sans.
  plt.savefig(
C:\Users\user\AppData\Local\Temp\ipykernel_8624\2373868330.py:26: UserWarning: Glyph 148 (\x94) missing from font(s) DejaVu Sans.
  plt.savefig(


✅ Saved above_average_sales.csv
✅ Saved above_average_sales.png


,Product Name,Sales
0,Bush Somerset Collection Bookcase,261.9600
1,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,Bretford CR4500 Series Slim Rectangular Table,957.5775
3,Mitel 5320 IP Phone VoIP phone,907.1520
4,Chromcraft Rectangular Conference Tables,1706.1840
...,...,...
2355,Avaya 5410 Digital phone,271.9600
2356,Cisco SPA301,249.5840
2357,Ibico Recycled Linen-Style Covers,437.4720
2358,Aastra 57i VoIP phone,258.5760


In [15]:
query = """
SELECT
Category,
ROUND(AVG(Discount),2) AS AvgDiscount
FROM Orders
GROUP BY Category;
"""
save_query_result(query,"avg_discount_category")
pd.read_sql(query, conn)

,Category,AvgDiscount
0,Furniture,0.17
1,Office Supplies,0.16
2,Technology,0.13


✅ Saved avg_discount_category.csv
✅ Saved avg_discount_category.png


,Category,AvgDiscount
0,Furniture,0.17
1,Office Supplies,0.16
2,Technology,0.13


In [16]:
query = """
SELECT
State,
ROUND(SUM(Profit),2) AS Profit
FROM Orders
GROUP BY State
ORDER BY Profit DESC
LIMIT 10;
"""
save_query_result(query,"top_states_profit")
pd.read_sql(query, conn)

,State,Profit
0,California,76381.39
1,New York,74038.55
2,Washington,33402.65
3,Michigan,24463.19
4,Virginia,18597.95
5,Indiana,18382.94
6,Georgia,16250.04
7,Kentucky,11199.70
8,Minnesota,10823.19
9,Delaware,9977.37


✅ Saved top_states_profit.csv
✅ Saved top_states_profit.png


,State,Profit
0,California,76381.39
1,New York,74038.55
2,Washington,33402.65
3,Michigan,24463.19
4,Virginia,18597.95
5,Indiana,18382.94
6,Georgia,16250.04
7,Kentucky,11199.70
8,Minnesota,10823.19
9,Delaware,9977.37


In [17]:
query = """
SELECT
City,
ROUND(SUM(Sales),2) AS Sales
FROM Orders
GROUP BY City
ORDER BY Sales DESC
LIMIT 10;
"""
save_query_result(query,"top_cities_sales")
pd.read_sql(query, conn)

,City,Sales
0,New York City,256368.16
1,Los Angeles,175851.34
2,Seattle,119540.74
3,San Francisco,112669.09
4,Philadelphia,109077.01
5,Houston,64504.76
6,Chicago,48539.54
7,San Diego,47521.03
8,Jacksonville,44713.18
9,Springfield,43054.34


✅ Saved top_cities_sales.csv
✅ Saved top_cities_sales.png


,City,Sales
0,New York City,256368.16
1,Los Angeles,175851.34
2,Seattle,119540.74
3,San Francisco,112669.09
4,Philadelphia,109077.01
5,Houston,64504.76
6,Chicago,48539.54
7,San Diego,47521.03
8,Jacksonville,44713.18
9,Springfield,43054.34


In [18]:
conn.close()